# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library. You will learn how to access metadata, enumerate data structures (record sets, fields, etc.), extract data, and perform exploratory analysis and visualizations.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"Version: {metadata['version']}, License: {metadata['license']}")
print(f"Published on: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs, using entity `@id`s for reference.

We'll explore the dataset's record sets and fields using the Croissant schema.

In [ ]:
# Explore record sets and their field definitions by @id
all_record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(all_record_sets)}\n")

record_set_ids = []
for record_set in all_record_sets:
    rs_id = record_set['@id']
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {record_set.get('name')}")
    print(f"  Description: {record_set.get('description')}")
    # List all fields (columns) for this record set
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/Columns:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id')}: {field.get('name')}")
        else:
            print(f"    - {field}")
    print()
if not record_set_ids:
    print("No top-level record sets discovered via the schema. Some datasets directly expose records without hierarchical record sets; see dataset documentation if so.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All extraction is done with reference to the record set and field `@id`s obtained from the previous overview.

We'll attempt to load each discovered record set. If no record sets are defined, try accessing default records (by field column IDs found in metadata, if applicable).

In [ ]:
# Attempt to extract data from discovered record sets
dataframes = {}
# If no record set IDs found, try to retrieve a default/only record set
if record_set_ids:
    ids_to_iterate = record_set_ids
else:
    ids_to_iterate = []
    # Try a default value based on documentation or metadata
    # (You may need to manually specify if the schema is not explicit)
    # ids_to_iterate = ['cr:RecordSet']
for record_set_id in ids_to_iterate:
    print(f"Extracting records for record set {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  No records found in {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head())

# If a main/default record set is used, assign it for reference
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field. We'll use a field's `@id` for all operations as required. 
Replace `<numeric_field_id>` with the `@id` (or column name) of a numeric field you want to explore (e.g., 'phq9_score', 'gad7_score', depending on previous code cell output). Similarly, for grouping, you can use a relevant category field like 'age_band' or 'gender'.

In [ ]:
import numpy as np
from IPython.display import display

# --- User Action: identify a numeric field ID and grouping field ID from the printed columns above ---
numeric_field_id = None
group_field_id = None
if main_record_set_id is not None:
    cols = dataframes[main_record_set_id].columns.tolist()
    # Attempt to auto-detect a numeric field for demo
    for col in cols:
        if 'score' in col or 'num' in col or 'age' in col:
            numeric_field_id = col
            break
    if not numeric_field_id:
        numeric_field_id = cols[0]  # fallback to the first column
    # Attempt to choose a grouping field
    prefer = ['age_band', 'gender', 'village', 'level_of_education', 'marital_status']
    for cand in prefer:
        if cand in cols:
            group_field_id = cand
            break
    if not group_field_id:
        group_field_id = cols[-1]  # fallback

    print(f"Numeric field selected: {numeric_field_id}")
    print(f"Grouping field selected: {group_field_id}")

    df = dataframes[main_record_set_id]

    # Drop missing values in chosen field for demonstration
    df_num = df.copy()
    df_num[numeric_field_id] = pd.to_numeric(df_num[numeric_field_id], errors='coerce')
    threshold = df_num[numeric_field_id].mean()
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalize the numeric field in the filtered sample
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by selected field and show mean
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)
else:
    print("No tabular record set available for EDA. Please check the previous cell's output and schema structure.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we will plot the histogram of the selected numeric field and a group-level summary chart.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    # Ensure numeric type
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id available, boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Visualization skipped -- no numeric or group field available.")

## 6. Conclusion
In this notebook, you have:
- Loaded the Croissant-based FAIR² Mental Health Survey dataset and explored its metadata
- Programmatically explored top-level record sets and columns using their `@id`s
- Loaded tabular data into dataframes and demonstrated typical EDA workflows (filtering, normalization, grouping)
- Created visualizations to illustrate data distributions and group differences

This dataset schema follows the Croissant FAIR-AI standard; you can now extend your exploration, train models, or join with other Croissant-compliant datasets as needed!